In [1]:
import os
import win32com.client as win32
from y_n import y_n
from upsilon_n import upsilon_n
import datetime
import time
import pandas as pd
import numpy as np
import random
from sklearn.cluster import MeanShift
from IPython.display import clear_output
from scipy.optimize import minimize

if __name__ == "__main__":

In [2]:
parafins = [ 'methane', 'ethane', 'propane', 
             'butane', 'pentane', 'hexane', 
               'heptane', 'octane', 
             'nonane', 'decane', 'undecane', 'dodecane',
               'tridecane', 'tetradecane', 'pentadecane', 'hexadecane', 'heptadecane',
               'octadecane', 'nonadecane', 'eicosane', 'heneicosane', 'docosane',
               'tricosane', 'tetracosane', 'pentacosane',
               'hexacosane', 'heptacosane',
               'octacosane', 'nonacosane', 'triacontane', 'hentriacontane', 'dotriacontane',
               'tritriacontane', 'tetratriacontane', 'pentatriacontane', 'hexatriacontane'
            ]

olefins = [ 'ethylene', 
            'propene',
            '1-butene', '1-pentene', '1-hexene',
            '1-heptene', '1-octene', 
            '1-nonene', '1-decene',
            '1-undecene', '1-dodecene',
            '1-tridecene', '1-tetradecene', '1-pentadecene',
            '1-hexadecene', '1-heptadecene', '1-octadecene',
          ]

others = ['carbon monoxide', 'hydrogen', 'water']

In [3]:
hysys = win32.Dispatch("HYSYS.Application")

In [4]:
hy_case = hysys.Application.ActiveDocument
hy_case.Visible = 1

In [5]:
hy_solver = hy_case.Solver

hy_solver.CanSolve = False

hy_f = hy_case.Flowsheet        

hy_ms = hy_f.MaterialStreams
hy_es = hy_f.EnergyStreams

reactor = hy_case.Flowsheet.Operations.Item('PFR-100')

In [6]:
n_parafins = {}
for i, parafin in enumerate(parafins):
    n = i + 1
    if n not in [4, 5, 6, 7, 8, 31, 32, 33, 34, 35, 36]:
        n_parafins[parafin] = n

n_olefins = {}
for i, olefin in enumerate(olefins):
    n = i + 2
    if n not in [3, 4, 5, 6, 7, 8]:
        n_olefins[olefin] = n

N = list(set(list(n_parafins.values()) + list(n_olefins.values())))

In [7]:
rxn_set = hy_case.BasisManager.ReactionPackageManager.ReactionSets.Item('FTS')

F1 = hy_ms.Item('F1')
F2 = hy_ms.Item('F2')
H2 = hy_ms.Item('H2')
CO = hy_ms.Item('CO')
C1_C3 = hy_ms.Item('C1-C3')
C9_C15 = hy_ms.Item('C9-C15')
C16p = hy_ms.Item('C16p')
unreacted_syngas = hy_ms.Item('unreacted syngas')
water = hy_ms.Item('water')

In [8]:
params = {'k_ads': np.float64(4.4879852669237044e-05),
 'H_ads': np.float64(0.006921867246030876),
 'A_HCs': np.float64(0.7771401117743281),
 'E_HCs': np.float64(119373.27942107256)}

In [9]:
overall_freq_fact = params['A_HCs'] # kmol / s / m³ cat / Pa²
activation_energy = params['E_HCs'] # J/mol
k_CO = params['k_ads'] # 1/Pa
H_CO = params['H_ads']
exp_CO_den = 1
exp_CO = 1
exp_H2 = 1

In [10]:
T = F2.TemperatureValue

alpha = float(np.load('param_alpha.npy'))

ALPHA_SHEET = hy_case.Flowsheet.Operations.Item('ALPHA')

ALPHA_SHEET.Cell(0,0).CellValue = alpha

# must multiply reaction rate by correction factor below
# because hysys bases reaction rate only on the gas phase, not the catalyst volume
vol_cat_to_vol_gas =  (1 / reactor.VoidFraction) - 1 # m³ cat / m³ gas
denominator = ((k_CO, H_CO, exp_CO_den, 0.0, 0.0, 0.0),
 (-32767.0, -32767.0, -32767.0, -32767.0, -32767.0, -32767.0))

for component, n in n_parafins.items():
    freq_fact = n * y_n(n) * upsilon_n(n, T) * overall_freq_fact * vol_cat_to_vol_gas # kmol / s / m³ gas / Pa²
    rx = rxn_set.ReactionPackage.Reactions.Item(component)
    idx_CO = rx.ReactantName().index('CO')
    idx_H2 = rx.ReactantName().index('Hydrogen')
    list_forward_orders = list(rx.ComponentForwardOrder())
    list_forward_orders[idx_H2] = exp_H2
    list_forward_orders[idx_CO] = exp_CO
    rx.ComponentForwardOrder = list_forward_orders
    rx.ForwardFrequencyFactor = freq_fact
    rx.ForwardActivationEnergy = activation_energy
    rx.DenominatorParametersValue = denominator
    # print(f'updating rate for {component}')

for component, n in n_olefins.items():
    freq_fact = n * y_n(n) * ( 1 - upsilon_n(n, T) ) * overall_freq_fact * vol_cat_to_vol_gas # kmol / s / m³ / Pa²
    rx = rxn_set.ReactionPackage.Reactions.Item(component)
    idx_CO = rx.ReactantName().index('CO')
    idx_H2 = rx.ReactantName().index('Hydrogen')
    list_forward_orders = list(rx.ComponentForwardOrder())
    list_forward_orders[idx_H2] = exp_H2
    list_forward_orders[idx_CO] = exp_CO
    rx.ComponentForwardOrder = list_forward_orders
    rx.ForwardFrequencyFactor = freq_fact
    rx.ForwardActivationEnergy = activation_energy
    rx.DenominatorParametersValue = denominator
    # print(f'updating rate for {component}')

# solve
# 3. Ligar o solver
hy_case.Solver.CanSolve = True

# 4. Aguardar a convergência
while hy_case.Solver.IsSolving:
    pass

hy_case.Solver.CanSolve = False